# Example 4: Creating a Coloured Globe (Complete Pipeline)

This notebook demonstrates the complete `globe3d` model generation pipeline:
1. Generate a hollow sphere with outer and inner meshes.
2. Apply ETOPO topography displacement to the outer shell.
3. Apply sharp coastline step boundary from a shapefile.
4. Assign vertex colors to the outer shell using a seismic tomography dataset.
5. Assign a solid, neutral gray color to the inner cavity vertices.
6. Configure magnets and export both hemispheres to OBJ files with vertex colors.

## Step 1: Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    LineDisplacer,
    GridColourer,
    ConstantColourer,
    calculate_displacement_scale
)

## Step 2: Generate hollow sphere

In [ ]:
model_radius_mm = 40.0

model = GlobeModel(
    n_points=8000,
    radius=model_radius_mm,
    hollow=True,
    inner_ratio=0.5,
)
print(f"Outer: {model.outer.vertices.shape[0]} vertices")
print(f"Inner: {model.inner.vertices.shape[0]} vertices")

## Step 3: Apply topography and coastline step

In [ ]:
# Load and downsample ETOPO topography
topo_grid = GeographicGrid.from_netcdf(
    "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc", 'lat', 'lon', 'z'
)
topo_grid_ds = GeographicGrid(
    lats=topo_grid.lats[::10],
    lons=topo_grid.lons[::10],
    grid=topo_grid.grid[::10, ::10]
)

# Displace topography (ETOPO data is in meters)
topo_units = 'm'
scale = calculate_displacement_scale(
    model_radius_mm, vertical_exagg=40.0, grid_units=topo_units
)
model.outer.displace(GridDisplacer(topo_grid_ds), scale=scale)

# Coastline step
model.outer.displace(LineDisplacer(
    shapefile_path="../inputs/coastlines/ne_110m_coastline.shp",
    displacement=0.8,
    width_degrees=0.5
))

## Step 4: Configure magnets

In [ ]:
model.configure_magnets(
    diameter=5.0,
    height=2.0,
    horizontal_tolerance=0.15,
    vertical_tolerance=0.10,
    vertical_offset=0.20,
    min_thickness=1.5,
    n_magnets=3,
    add_bosses=True,
)

## Step 5: Apply surface coloring

We color the outward-facing surfaces from a seismic tomography grid, and paint the inward-facing cavity surfaces a neutral grey.

In [ ]:
# Load tomography grid
tomo_grid = GeographicGrid.from_netcdf(
    "../inputs/s40_depth_slice_2850.grd", 'y', 'x', 'z'
)

# 1. Color outer (outward-facing) surfaces using tomography grid
model.outer.colour(
    GridColourer(tomo_grid, colormap='RdBu_r', vmin=-2.0, vmax=2.0),
    selection='outward_facing',
)

# 2. Color inner (inward-facing) cavity surfaces to gray
model.outer.colour(
    ConstantColourer([0.6, 0.6, 0.6]),
    selection='inward_facing',
)

## Step 6: Preview colors in 3D

In [ ]:
# Generate hemispheres for preview
top_half, bottom_half = model.generate_hemispheres(engine='manifold')

fig = plt.figure(figsize=(12, 6))

top_colors = top_half.visual.vertex_colors[:, :3].astype(float) / 255.0
bottom_colors = bottom_half.visual.vertex_colors[:, :3].astype(float) / 255.0

ax1 = fig.add_subplot(121, projection='3d')
pts_top = top_half.vertices
ax1.scatter(pts_top[:, 0], pts_top[:, 1], pts_top[:, 2], c=top_colors, s=2)
ax1.set_title("Colored Top Hemisphere")

ax2 = fig.add_subplot(122, projection='3d')
pts_bot = bottom_half.vertices
ax2.scatter(pts_bot[:, 0], pts_bot[:, 1], pts_bot[:, 2], c=bottom_colors, s=2)
ax2.set_title("Colored Bottom Hemisphere")

plt.show()

## Step 7: Export to OBJ with vertex colors

In [ ]:
model.export_hemispheres(
    "../outputs/example_4_top.obj",
    "../outputs/example_4_bottom.obj",
    engine='manifold',
)
print("Colored OBJ hemispheres exported successfully!")